In [35]:
import numpy as np
import tensorflow as tf
import string

from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [36]:
# Load IMDB word index

max_len = 500
num_words = 10000

In [37]:
# Building the  Model 

model = Sequential()
model.add(Embedding(num_words, 128, input_length=max_len)) ## Embedding layer
model.add(SimpleRNN(128, activation="relu")) ## Simple RNN layer
model.add(Dense(1, activation="sigmoid")) ## Output layer
model.build(input_shape=(None, max_len))

In [38]:
# Load trained weights
model.load_weights("model.weights.h5")
model.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 500, 128)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_4 (SimpleRNN)        │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,025 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

In [39]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

In [40]:
# Helper Function: Decode Review

def decode_review(encoded_review):
    return " ".join([reverse_word_index.get(i - 3, "?") for i in encoded_review])

In [41]:
# CORRECT Preprocessing Function

def preprocess_text(text):
    # Lowercase
    text = text.lower()
    # Remove punctuation (CRITICAL FIX)
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = text.split()
    encoded_review = []
    for word in words:
        index = word_index.get(word)
        # Match training vocab limit
        if index is not None and index < num_words:
            encoded_review.append(index + 3)
        else:
            encoded_review.append(2)  # OOV token

    padded_review = sequence.pad_sequences([encoded_review], maxlen=max_len)

    return padded_review

In [42]:
# Prediction Function

def predict_sentiment(review):
    preprocessed_input = preprocess_text(review)
    prediction = model.predict(preprocessed_input, verbose=0)
    score = float(prediction[0][0])
    sentiment = "Positive" if score >= 0.5 else "Negative"
    return sentiment, score

In [43]:
# Example Prediction

example_review = "This movie was worst and boring. I hated it."

sentiment, score = predict_sentiment(example_review)
print("Review:", example_review)
print("Sentiment:", sentiment)
print("Prediction Score:", round(score, 4))

Review: This movie was worst and boring. I hated it.
Sentiment: Negative
Prediction Score: 0.3746
